In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import urllib.request

from model import Iteration_Model 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [2]:
url = "https://www.gutenberg.org/cache/epub/1661/pg1661.txt"
req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})

with urllib.request.urlopen(req) as response:
    raw_text = response.read().decode("utf-8")

# Strip the Project Gutenberg header and footer
start_idx = raw_text.find("I. A SCANDAL IN BOHEMIA")
end_idx = raw_text.find("End of the Project Gutenberg")
text = raw_text[start_idx:end_idx].strip()

print(f"Loaded {len(text)} characters of normal English text.")
print(f"Sample preview:\n{text[:200]}")

Loaded 592375 characters of normal English text.
Sample preview:
I. A SCANDAL IN BOHEMIA


I.

To Sherlock Holmes she is always _the_ woman. I have seldom heard him
mention her under any other name. In his eyes she eclipses and
predominates the whole of her 


In [3]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

char_to_id = {ch: i for i, ch in enumerate(chars)}
id_to_char = {i: ch for i, ch in enumerate(chars)}

def encode(string):
    return [char_to_id[ch] for ch in string]

def decode(indices):
    return "".join([id_to_char[idx] for idx in indices])

# Convert entire book into a single tensor
data = torch.tensor(encode(text), dtype=torch.long, device=device)
print(f"Vocab size: {vocab_size} unique characters")

Vocab size: 97 unique characters


In [4]:
def get_batch(batch_size=16, seq_len=256):
    # Grab longer text chunks so the window has room to slide
    max_start = len(data) - seq_len
    start_indices = torch.randint(0, max_start, (batch_size,))
    
    sequences = torch.stack([data[i:i + seq_len] for i in start_indices])
    return sequences

In [5]:
def inspect_window(data):
    if data['window_idx'] == 0:
        raw_loss = data['raw_loss']
        slot_losses = raw_loss.mean(dim=0).tolist()
        formatted = [round(l, 2) for l in slot_losses]
        print(f"Slot losses: {formatted}")

        # 1. Decode the preceding context window that the model just read
        input_ids = data['input'][0].tolist()
        context_text = decode(input_ids)

        # 2. Decode the highest probability token for each future slot
        pred_ids = torch.argmax(data['predictions'][0], dim=-1).tolist()
        pred_text = decode(pred_ids)

        # 3. Decode the actual ground truth tokens that came next
        target_ids = data['targets'][0].tolist()
        target_text = decode(target_ids)

        # 4. Print the trailing part of context along with the predictions
        # Slicing the last 48 characters keeps the output easy to read in the terminal
        print(f"Context:        {repr(context_text)}")
        print(f"Pred:           {repr(pred_text)}")
        print(f"Target:         {repr(target_text)}")

In [6]:
# Streaming callback for real-time generation preview
def stream_character(data):
    char = decode(data['new_tokens'][0].tolist())
    
    # Swap raw line breaks with visible text markers
    safe_char = char.replace('\n', '\\n').replace('\r', '\\r')
    
    print(safe_char, end="", flush=True)

In [ ]:
# Window and sequence configuration
window_size = 128
stride = 1
num_predictions = 16
num_windows = 4

seq_len = window_size + num_predictions + (num_windows - 1) * stride
batch_size = 32
num_steps = 1000

# Initialize model without hardcoded stride or prediction horizons
model = torch.compile(Iteration_Model(
    vocab_size=vocab_size,
    hidden_dim=256,
    num_heads=8,
    num_layers=4,
    sweeps=2,
    sweep_iters=2,
    layer_iters=2,
    window_size=window_size
).to(device))

optimizer = optim.AdamW(model.parameters(), lr=0.001)

for step in range(1, num_steps + 1):
    batch = get_batch(batch_size=batch_size, seq_len=seq_len)
    
    # Attach diagnostic callback only on evaluation checkpoints
    cb = inspect_window if (step % 100 == 0 or step == 1) else None

    # train_model handles window slicing, gradient accumulation, and decayed loss internally
    avg_loss, _ = model.train_model(
        sequence=batch,
        optimizer=optimizer,
        stride=stride,
        num_predictions=num_predictions,
        prediction_decay=True,
        min_pred_decay=0.25,
        accumulate_gradients=True,
        on_window=cb
    )

    if step % 100 == 0 or step == 1:
        print(f"Step {step:04d} | Window Avg Loss {avg_loss:.4f}")
        
        prompt_text = "The man was "
        prompt_tokens = torch.tensor([encode(prompt_text)], dtype=torch.long, device=device)
        
        print(f"Sample: '{prompt_text}", end="", flush=True)
        model.generate(
            prompt=prompt_tokens,
            num_tokens=64,
            temperature=0.8,
            stride=1,
            num_predictions=num_predictions,
            on_step=stream_character
        )
        print("'\n")

Slot losses: [4.6, 4.63, 4.73, 4.87, 4.51, 4.61, 4.8, 4.73]
Context:        ' his bills were all paid in ready money,\r\nand he had no friends '
Pred:           ' g5$vHé&'
Target:         'of any s'
Step 0001 | Window Avg Loss 4.7092
Sample: 'The man was j . ;’8 J’s  $#\rza:e œ? tteb7t  à£6e™  KeFegsànlrb,$  4é-FQ3BTs '

Slot losses: [2.99, 3.72, 3.17, 3.03, 3.18, 3.28, 3.13, 3.18]
Context:        's breathing now.”\r\n\r\n“He can’t lie in the street. May we bring h'
Pred:           'e       '
Target:         'im in, m'
Step 0100 | Window Avg Loss 3.2063
Sample: 'The man was Gd t iwes ce pet thy w”\r\nwicothen\r\nx Mr whin wo os \nE it tl won,'

Slot losses: [2.41, 3.1, 3.09, 2.86, 3.12, 3.37, 3.17, 2.99]
Context:        ' garden behind\r\ninto the stable lane. So long was he that Mr. Ho'
Pred:           'u       '
Target:         'lder and'
Step 0200 | Window Avg Loss 3.0012
Sample: 'The man was nd\r\n“As s aderea abe te whiks my g at te, wang s. t sese m. aly '

Slot losses: [1.83, 2

In [8]:
for name, param in model.named_parameters():
    if param.grad is not None:
        print(f"{name:25s} | grad norm: {param.grad.norm().item():.8f}")